# IR Assignment 1 — Text Preprocessing & Inverted Index
## End-to-End Information Retrieval System

**Course:** AIMLCZG537 / DSECLZG537 — Information Retrieval, BITS Pilani WILP, Semester 2 2025-26  

---

**Group 64**

| Name | BITS ID | Contribution |
|------|---------|---------------|
| Prithvi Mohanty | 2025AA05707 | 33% |
| Monika Sharma | *(fill in)* | 33% |
| Hanni Rajavikra... | *(fill in)* | 34% |

---

> **Dataset:** Built-in COVID-19 vaccine research excerpts (Docs 0–3) and IR/NLP reference
> documents (Docs 4–7). Reproduced inline — no external file dependency.

This notebook demonstrates the complete preprocessing pipeline and inverted index construction
that underpins the Streamlit IR application.


In [ ]:
import re
import math
from collections import defaultdict

import nltk
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'averaged_perceptron_tagger']:
    nltk.download(pkg, quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords as _sw_corpus
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
from nltk import pos_tag
import pandas as pd

print('Libraries loaded.')

## 1. Document Corpus

We work with 8 documents: 4 COVID-19 vaccine research excerpts and 4 IR/NLP/ML concept documents.


In [ ]:
CORPUS = {
    0: ('COVID Vaccine Barriers — Ghana Qualitative Study',
        'Facilitators and barriers to COVID-19 vaccine uptake among women in two regions of Ghana. '
        'Several studies have found gender differences in vaccine uptake, with women less likely to vaccinate. '
        'Key facilitators include desire to protect family, education about vaccines, and vaccines being cost-free. '
        'Long queues, fear of side effects, misconceptions, and shortage of vaccines were main barriers.'),
    1: ('COVID Vaccine Acceptance — Ghana Determinants',
        'The level and determinants of COVID-19 vaccine acceptance in Ghana. '
        'Determinants of acceptance included trust in health authorities, perceived vaccine safety, and vaccination history. '
        'Barriers included fear of adverse effects, religious beliefs, and distrust of the vaccination campaign.'),
    2: ('COVID Vaccine Misinformation and Hesitancy',
        'Misinformation of COVID-19 vaccines and vaccine hesitancy. '
        'Misinformation significantly increased vaccine hesitancy even after correction attempts. '
        'Social media platforms amplified the spread of misinformation, reducing community confidence.'),
    3: ('COVID Vaccine Confidence, Effectiveness and Safety',
        'Confidence in COVID-19 vaccine effectiveness and safety and its effect on uptake decisions. '
        'Vaccine confidence encompasses beliefs about safety, effectiveness, and reliability of health systems. '
        'Higher confidence in vaccine effectiveness correlates strongly with willingness to vaccinate.'),
    4: ('Information Retrieval — Core Concepts',
        'Information retrieval systems enable users to find relevant documents from large text collections. '
        'The inverted index maps each vocabulary term to the list of documents containing that term. '
        'TF-IDF weighting assigns higher importance to terms frequent in one document but rare across the corpus. '
        'Inverted index lookup is efficient. Lookup queries retrieve matching documents.'),
    5: ('NLP Preprocessing Pipeline',
        'Natural language processing enables computers to understand and generate human language. '
        'Tokenization splits text into atomic units before any further linguistic processing. '
        'Stemming reduces words to their root form using rule-based heuristic truncation. '
        'Lemmatization uses morphological analysis to find the canonical base form of each word.'),
    6: ('IR Dictionary Structures and Tolerant Retrieval',
        'Dictionary data structures store the vocabulary of an information retrieval system for efficient lookup. '
        'Binary search trees support O(log n) average-case search but degenerate to O(n) with sorted key insertion. '
        'Wildcard queries use k-gram indices to find vocabulary terms matching incomplete spellings.'),
    7: ('Web Search and Machine Learning Ranking',
        'Search engines use crawling, indexing, and ranking algorithms to retrieve relevant web pages. '
        'Machine learning algorithms improve ranking by learning relevance patterns from labelled query-document pairs. '
        'Deep learning models such as BERT have achieved state-of-the-art results on IR benchmarks.'),
}

print(f'{len(CORPUS)} documents loaded.')
for doc_id, (title, _) in CORPUS.items():
    print(f'  Doc {doc_id}: {title}')

## 2. Preprocessing Pipeline

We apply the following steps in order:

1. **Hyphen handling** — replace `-` with space so compound words become separate tokens
2. **Lowercasing** — normalize all surface variants
3. **Punctuation removal** — strip non-alphanumeric characters
4. **Tokenization** — `nltk.word_tokenize`
5. **Stop word removal** — filter NLTK English stop words (179 words)
6. **Stemming / Lemmatization** — configurable


In [ ]:
STOPWORDS  = set(_sw_corpus.words('english'))
_porter    = PorterStemmer()
_snowball  = SnowballStemmer('english')
_lemma     = WordNetLemmatizer()

def _wn_pos(tag):
    if tag.startswith('J'): return 'a'
    if tag.startswith('V'): return 'v'
    if tag.startswith('R'): return 'r'
    return 'n'

def preprocess(text, lowercase=True, remove_sw=True, split_hyphens=True, method='None'):
    if split_hyphens:
        text = text.replace('-', ' ')
    if lowercase:
        text = text.lower()
    text   = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if len(t) > 1]
    if remove_sw:
        tokens = [t for t in tokens if t not in STOPWORDS]
    if method == 'Porter Stemmer':
        tokens = [_porter.stem(t) for t in tokens]
    elif method == 'Snowball Stemmer':
        tokens = [_snowball.stem(t) for t in tokens]
    elif method == 'Lemmatizer':
        tagged = pos_tag(tokens)
        tokens = [_lemma.lemmatize(w, _wn_pos(tag)) for w, tag in tagged]
    return tokens

# Quick test
sample = CORPUS[0][1]
raw_tokens = word_tokenize(sample.lower())
proc_tokens = preprocess(sample)
print(f'Raw token count : {len(raw_tokens)}')
print(f'After preprocessing: {len(proc_tokens)}')
print(f'Sample tokens: {proc_tokens[:10]}')

### 2.1 Before vs After Preprocessing (first 3 documents)


In [ ]:
for doc_id in range(3):
    title, text = CORPUS[doc_id]
    original = text[:120] + '...'
    processed = preprocess(text)
    print(f'Doc {doc_id}: {title}')
    print(f'  Original  : {original}')
    print(f'  Processed : {processed[:12]}')
    print()

### 2.2 Vocabulary Size Before vs After Preprocessing


In [ ]:
from nltk.tokenize import word_tokenize

raw_vocab = set()
proc_vocab = set()

for _, text in CORPUS.values():
    raw_vocab  |= set(w.lower() for w in word_tokenize(text) if len(w) > 1)
    proc_vocab |= set(preprocess(text))

reduction = (1 - len(proc_vocab) / len(raw_vocab)) * 100
print(f'Raw vocabulary size    : {len(raw_vocab)}')
print(f'Preprocessed vocabulary: {len(proc_vocab)}')
print(f'Vocabulary reduction   : {reduction:.1f}%')

df_vocab = pd.DataFrame({
    'Metric': ['Raw vocabulary', 'After preprocessing', 'Reduction'],
    'Value' : [len(raw_vocab), len(proc_vocab), f'{reduction:.1f}%']
})
print(df_vocab.to_string(index=False))

## 3. Inverted Index Construction

An inverted index maps each vocabulary term to the sorted list of document IDs that contain it.
It is the core data structure enabling fast Boolean and phrase retrieval.


In [ ]:
def build_inverted_index(docs):
    idx = defaultdict(set)
    for doc_id, tokens in docs.items():
        for tok in tokens:
            idx[tok].add(doc_id)
    return {k: sorted(v) for k, v in sorted(idx.items())}

processed_docs = {doc_id: preprocess(text) for doc_id, (_, text) in CORPUS.items()}
inv_index = build_inverted_index(processed_docs)

print(f'Vocabulary size: {len(inv_index)} terms')
print('\nFirst 20 index entries:')
df_idx = pd.DataFrame(
    [{'Term': t, 'Doc Freq': len(docs), 'Postings': str(docs)}
     for t, docs in list(inv_index.items())[:20]]
)
print(df_idx.to_string(index=False))

## 4. Stemming vs Lemmatization

We compare Porter Stemmer, Snowball Stemmer, and WordNet Lemmatizer on representative words
from the corpus vocabulary.


In [ ]:
test_words = ['vaccines', 'running', 'barriers', 'studies', 'universities',
              'algorithms', 'misinformation', 'effectiveness', 'retrieval']

rows = []
for w in test_words:
    tagged = pos_tag([w])
    rows.append({
        'Word'           : w,
        'Porter Stem'    : _porter.stem(w),
        'Snowball Stem'  : _snowball.stem(w),
        'Lemma (WordNet)': _lemma.lemmatize(w, _wn_pos(tagged[0][1])),
    })

df_compare = pd.DataFrame(rows)
print(df_compare.to_string(index=False))

### 4.1 Retrieval Quality Comparison — Jaccard Similarity

We run 5 test queries using OR-retrieval under both stemming and lemmatization, then compute
Jaccard similarity between the result sets to measure retrieval agreement.


In [ ]:
TEST_QUERIES = [
    'vaccine barriers',
    'information retrieval',
    'machine learning ranking',
    'misinformation hesitancy',
    'language processing tokenization',
]

# Build stem and lemma indices
proc_stem  = {d: preprocess(t, method='Porter Stemmer')  for d, (_, t) in CORPUS.items()}
proc_lemma = {d: preprocess(t, method='Lemmatizer')      for d, (_, t) in CORPUS.items()}
idx_stem   = build_inverted_index(proc_stem)
idx_lemma  = build_inverted_index(proc_lemma)

rows, jaccards = [], []
for q in TEST_QUERIES:
    stem_q  = preprocess(q, method='Porter Stemmer')
    lemma_q = preprocess(q, method='Lemmatizer')
    stem_docs  = set().union(*(set(idx_stem.get(t, [])) for t in stem_q))
    lemma_docs = set().union(*(set(idx_lemma.get(t, [])) for t in lemma_q))
    inter = stem_docs & lemma_docs
    union = stem_docs | lemma_docs
    j = len(inter) / len(union) if union else 1.0
    jaccards.append(j)
    rows.append({'Query': q, 'Stem docs': sorted(stem_docs),
                 'Lemma docs': sorted(lemma_docs), 'Jaccard': round(j, 2)})

df_jaccard = pd.DataFrame(rows)
print(df_jaccard.to_string(index=False))
print(f'\nAverage Jaccard similarity: {sum(jaccards)/len(jaccards):.2f}')

### 4.2 Analysis: Which Method Is Better for This Corpus?

**Stemming** (Porter/Snowball) applies rule-based suffix truncation:
- *vaccines* → *vaccin*, *universities* → *univers*, *barriers* → *barrier*
- Higher recall (more aggressive matching) but produces non-dictionary tokens

**Lemmatization** (WordNet) uses morphological analysis:
- *vaccines* → *vaccine*, *universities* → *university*, *barriers* → *barrier*
- Slightly lower recall but preserves valid dictionary forms

**Conclusion:** For this COVID-19 / IR domain corpus, **lemmatization is preferred**.
The corpus contains domain-specific nouns (*vaccine*, *retrieval*, *hesitancy*) where
over-stemming introduces vocabulary noise — *vaccin* is not a valid token and could
cause vocabulary fragmentation in a larger system.
Average Jaccard similarity across 5 queries quantifies their retrieval overlap.


## 5. Boolean Query on Inverted Index

Demonstrating AND and OR queries on the inverted index.


In [ ]:
def and_query(terms, index):
    if not terms: return set()
    result = set(index.get(terms[0], []))
    for t in terms[1:]:
        result &= set(index.get(t, []))
    return result

def or_query(terms, index):
    result = set()
    for t in terms:
        result |= set(index.get(t, []))
    return result

# Example: AND query
q_terms = [preprocess(w)[0] if preprocess(w) else w
           for w in ['vaccine', 'hesitancy'] if preprocess(w)]
print(f'AND query terms : {q_terms}')
print(f'AND result docs : {sorted(and_query(q_terms, inv_index))}')

or_terms = [preprocess(w)[0] if preprocess(w) else w
            for w in ['retrieval', 'vaccine'] if preprocess(w)]
print(f'\nOR query terms  : {or_terms}')
print(f'OR result docs  : {sorted(or_query(or_terms, inv_index))}')